# EditGuard: inspect the actual Colab runtime

In Colab, select **Runtime → Change runtime type → GPU**, then run the cell below.
The assigned GPU matters: do not assume every Colab session supplies the same device.

This cell installs nothing, downloads no model weights, and runs no training.
It records GPU memory, host RAM, disk capacity and installed package versions.
**This is not an editing benchmark, fit guarantee, or performance result.**
The finite stopping condition is saving one JSON inventory. The GPU query has a 20-second timeout.

Keep the generated JSON file for the next setup/profile step. The existing local website is unaffected.


In [ ]:
import csv
import importlib.metadata
import io
import json
import os
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def host_memory():
    if Path("/proc/meminfo").exists():
        values = {}
        for line in Path("/proc/meminfo").read_text().splitlines():
            key, value = line.split(":", 1)
            if key in ("MemTotal", "MemAvailable"):
                values[key] = int(value.strip().split()[0]) * 1024
        return values
    if os.name == "nt":
        import ctypes

        class MemoryStatus(ctypes.Structure):
            _fields_ = [
                ("length", ctypes.c_ulong), ("load", ctypes.c_ulong),
                ("total_physical", ctypes.c_ulonglong),
                ("available_physical", ctypes.c_ulonglong),
                ("total_page_file", ctypes.c_ulonglong),
                ("available_page_file", ctypes.c_ulonglong),
                ("total_virtual", ctypes.c_ulonglong),
                ("available_virtual", ctypes.c_ulonglong),
                ("available_extended_virtual", ctypes.c_ulonglong),
            ]

        state = MemoryStatus()
        state.length = ctypes.sizeof(MemoryStatus)
        if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(state)):
            return {"MemTotal": state.total_physical,
                    "MemAvailable": state.available_physical}
    return {"status": "unavailable"}


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


gpus = []
gpu_inventory_error = None
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=20,
    )
    for row in csv.reader(io.StringIO(result.stdout), skipinitialspace=True):
        if row:
            gpus.append({"name": row[0].strip(),
                         "total_memory_mib": int(row[1].strip()),
                         "free_memory_mib": int(row[2].strip()),
                         "driver_version": row[3].strip()})
except (OSError, subprocess.SubprocessError, ValueError, IndexError) as error:
    gpu_inventory_error = type(error).__name__ + ": " + str(error)

disk = shutil.disk_usage(Path.cwd())
now = datetime.now(timezone.utc)
report = {
    "purpose": "hardware_inventory_only",
    "captured_at_utc": now.isoformat(),
    "runtime_location": "Colab" if Path("/content").is_dir() else "local_or_other",
    "runs_training": False,
    "downloads_models": False,
    "installs_packages": False,
    "python_version": platform.python_version(),
    "operating_system": platform.system(),
    "gpus": gpus,
    "gpu_inventory_error": gpu_inventory_error,
    "host_memory_bytes": host_memory(),
    "disk_bytes": {"total": disk.total, "free": disk.free},
    "installed_versions": {
        name: package_version(name)
        for name in ("torch", "transformers", "higher", "hydra-core")
    },
    "interpretation": (
        "Inventory only: this does not demonstrate that any editor fits or runs. "
        "The next step is selecting a compatible isolated environment and "
        "profiling one real edit on this exact runtime."
    ),
}
destination = Path("editguard-runtime-inventory-" + now.strftime("%Y%m%dT%H%M%SZ") + ".json")
destination.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
print("\nSaved inventory:", destination.name)
